<a href="https://colab.research.google.com/github/ah1ahwon/seoul_mobility/blob/main/Seoul_Mobility_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seoul 2030 Mobility — 전체 파이프라인 (다운로드 → 분석 → 시각화)

이 노트북은 Google Colab에서 **처음부터 끝까지** 실행하는 파이프라인입니다.

| 단계 | 내용 |
|---|---|
| Step 1 | Google Drive 마운트 |
| Step 2 | GitHub 코드 클론 |
| Step 3 | 패키지 설치 (필수 + GIS) |
| Step 4 | 데이터 직접 다운로드 → Drive 저장 |
| Step 4-5 | 필수 데이터 다운로드 (매출·생활인구) |
| Step 5 | 분석 실행 |
| Step 6 | 결과 Drive 저장 |
| Step 7 | 시각화 (이동 추세 + 방문 패턴 + 법정동) |

**소요 시간 예상:** 다운로드 ~60분 + 분석 ~20분 + 시각화 ~5분

**Drive 필요 용량:** 약 5~6GB (월말 스냅샷 39개 + 3월 일별 30개 + 보조 파일)

> **필수 레이어**: 매출·생활인구·GIS·법정동·교통 좌표 데이터가 모두 있어야 최종 분석이 완료됩니다. 개발용 부분 실행이 필요할 때만 `SEOUL_ALLOW_PARTIAL=1`을 사용합니다.


## Step 1. Google Drive 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ▼ Drive 내 저장 폴더 (필요시 변경)
DRIVE_BASE   = Path("/content/drive/MyDrive/seoul_mobility")
DRIVE_RAW    = DRIVE_BASE / "raw"
DRIVE_OUTPUT = DRIVE_BASE / "output"

DRIVE_RAW.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Drive 경로 준비 완료:", DRIVE_RAW)

Mounted at /content/drive
Drive 경로 준비 완료: /content/drive/MyDrive/seoul_mobility/raw


## Step 2. GitHub에서 코드 클론

In [3]:
from pathlib import Path

REPO_URL = 'https://github.com/ah1ahwon/seoul_mobility.git'
REPO_DIR = '/content/seoul_mobility'

if Path(REPO_DIR).exists():
    !git -C {REPO_DIR} fetch origin --quiet
    !git -C {REPO_DIR} reset --hard origin/main --quiet
    print('최신 코드로 강제 업데이트 완료')
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}
    print('코드 클론 완료')

%cd {REPO_DIR}
!git log --oneline -1
!wc -l seoul_mobility_analysis.py

# ⚠️  노트북 셀이 GitHub 최신 버전과 다를 수 있습니다.
# Drive에 저장된 이전 버전을 실행 중이라면 아래 링크에서 새로 열어주세요:
# https://colab.research.google.com/github/ah1ahwon/seoul_mobility/blob/main/Seoul_Mobility_Full_Pipeline.ipynb
print('─' * 60)
print('위 git log가 최신 커밋이고, seoul_mobility_analysis.py가 2700줄대이면 최신 코드입니다.')
print('노트북을 항상 최신 버전으로 실행하려면 위 링크를 사용하세요.')
print('─' * 60)


코드 클론 완료
/content/seoul_mobility


## Step 3. 패키지 설치

In [4]:
!pip install -q -r requirements.txt
print("기본 패키지 설치 완료")

!pip install -q geopandas shapely
print("GIS 패키지 설치 완료")

패키지 설치 완료


## Step 4. 데이터 다운로드 → Google Drive 저장

서울 공공데이터에서 직접 Drive로 다운로드합니다. 이미 있는 파일은 건너뜁니다.

- 생활이동 월말 스냅샷 39개 (2023-01 ~ 2026-03)
- 2026년 3월 일별 30개
- 성·연령별 도착지 기준 월별 파일 (`seoul_purpose_admdong1_in_YYYYMM.zip`)
- 보조 데이터 4종 (지하철, 버스, 행정동, 시민생활)

In [ ]:
import subprocess
import time
import csv
import zipfile as _zipfile

REFERER_LIVING = "https://data.seoul.go.kr/dataList/OA-22299/F/1/datasetView.do"
DL_URL = "https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?useCache=false"

def _is_valid_zip(path) -> bool:
    try:
        with _zipfile.ZipFile(path):
            return True
    except _zipfile.BadZipFile:
        return False

def download_file(inf_id, inf_seq, file_seq, output_path, referer, label="", is_zip=None):
    """서울 공공데이터 단일 파일 다운로드. 이미 존재하고 유효하면 건너뜀."""
    out = Path(output_path)
    check_zip = is_zip if is_zip is not None else out.suffix.lower() == ".zip"

    if out.exists() and out.stat().st_size > 100_000:
        if check_zip and not _is_valid_zip(out):
            print(f"  re-download (corrupt ZIP): {out.name}")
            out.unlink()
        else:
            print(f"  skip (exists): {out.name}")
            return True

    print(f"  downloading: {label or out.name} ...", end=" ", flush=True)
    result = subprocess.run(
        [
            "curl", "-L", "-s",
            "-e", referer,
            "-X", "POST",
            "-d", f"infId={inf_id}",
            "-d", f"infSeq={inf_seq}",
            "-d", f"seq={file_seq}",
            "-d", f"seqNo={file_seq}",
            DL_URL,
            "-o", str(out),
        ],
        capture_output=True,
    )
    if result.returncode != 0 or not out.exists() or out.stat().st_size < 100_000:
        print("FAIL (download error)")
        out.unlink(missing_ok=True)
        return False
    if check_zip and not _is_valid_zip(out):
        print("FAIL (server returned error page, not a valid ZIP)")
        out.unlink(missing_ok=True)
        return False
    size_mb = out.stat().st_size / 1_048_576
    print(f"OK ({size_mb:.1f} MB)")
    return True

print("다운로드 함수 준비 완료")

In [6]:
# ── 4-1. 보조 데이터 4종 ──────────────────────────────────────────────
print("=== 보조 데이터 다운로드 ===")

aux_files = [
    ("OA-12914", "3", "153", DRIVE_RAW / "CARD_SUBWAY_MONTH_202604.csv",
     "https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do", "지하철 2026-04"),
    ("OA-12913", "3", "109", DRIVE_RAW / "bus_time_station_202604.csv",
     "https://data.seoul.go.kr/dataList/OA-12913/S/1/datasetView.do", "버스 2026-04"),
    ("OA-22160", "3", "1",   DRIVE_RAW / "seoul_admin_dong_area.zip",
     "https://data.seoul.go.kr/dataList/OA-22160/S/1/datasetView.do", "행정동 매핑"),
    ("OA-22266", "1", "30",  DRIVE_RAW / "seoul_living_interest_groups_202512.xlsx",
     "https://data.seoul.go.kr/dataList/OA-22266/F/1/datasetView.do", "시민생활 2025-12"),
]

for inf_id, inf_seq, file_seq, out_path, ref, label in aux_files:
    download_file(inf_id, inf_seq, file_seq, out_path, ref, label)
    time.sleep(1)

print("\n보조 데이터 완료")

=== 보조 데이터 다운로드 ===
  skip (exists): CARD_SUBWAY_MONTH_202604.csv
  skip (exists): bus_time_station_202604.csv
  skip (exists): seoul_admin_dong_area.zip
  skip (exists): seoul_living_interest_groups_202512.xlsx

보조 데이터 완료


In [7]:
from pathlib import Path
# ── 4-2. 생활이동 월말 스냅샷 39개 (2023-01 ~ 2026-03) ──────────────
import csv as _csv

MANIFEST_PATH = Path("/content/seoul_mobility/data_archive/metadata/living_migration_month_end_manifest.csv")

with MANIFEST_PATH.open() as f:
    reader = _csv.DictReader(f)
    month_end_entries = list(reader)

print(f"=== 월말 스냅샷 {len(month_end_entries)}개 다운로드 ===")
failed = []
for i, row in enumerate(month_end_entries, 1):
    yyyymm   = row["yyyymm"]
    filename = row["filename"]
    seq      = row["seq"]
    out_path = DRIVE_RAW / filename
    print(f"[{i:02d}/{len(month_end_entries)}] {yyyymm}", end="  ")
    ok = download_file("OA-22299", "1", seq, out_path, REFERER_LIVING, filename)
    if not ok:
        failed.append(filename)
    time.sleep(0.5)

print(f"\n월말 스냅샷 완료. 실패: {len(failed)}개")
if failed:
    print("실패 파일:", failed)

=== 월말 스냅샷 39개 다운로드 ===
[01/39] 202301    skip (exists): seoul_purpose_admdong4_in_20230131.zip
[02/39] 202302    skip (exists): seoul_purpose_admdong4_in_20230228.zip
[03/39] 202303    skip (exists): seoul_purpose_admdong4_in_20230331.zip
[04/39] 202304    skip (exists): seoul_purpose_admdong4_in_20230430.zip
[05/39] 202305    skip (exists): seoul_purpose_admdong4_in_20230531.zip
[06/39] 202306    skip (exists): seoul_purpose_admdong4_in_20230630.zip
[07/39] 202307    skip (exists): seoul_purpose_admdong4_in_20230731.zip
[08/39] 202308    skip (exists): seoul_purpose_admdong4_in_20230831.zip
[09/39] 202309    skip (exists): seoul_purpose_admdong4_in_20230930.zip
[10/39] 202310    skip (exists): seoul_purpose_admdong4_in_20231031.zip
[11/39] 202311    skip (exists): seoul_purpose_admdong4_in_20231130.zip
[12/39] 202312    skip (exists): seoul_purpose_admdong4_in_20231231.zip
[13/39] 202401    skip (exists): seoul_purpose_admdong4_in_20240131.zip
[14/39] 202402    skip (exists): seoul_p

In [8]:
# ── 4-3. 2026년 3월 일별 30개 ────────────────────────────────────────
import datetime

print("=== 2026년 3월 일별 다운로드 ===")
start = datetime.date(2026, 3, 1)
end   = datetime.date(2026, 3, 31)
skip  = {datetime.date(2026, 3, 28)}  # 원천 목록에 없음

daily_dates = [
    start + datetime.timedelta(days=d)
    for d in range((end - start).days + 1)
    if (start + datetime.timedelta(days=d)) not in skip
]

failed_daily = []
for i, dt in enumerate(daily_dates, 1):
    seq      = dt.strftime("%y%m%d")           # e.g. 260301
    filename = f"seoul_purpose_admdong4_in_{dt.strftime('%Y%m%d')}.zip"
    out_path = DRIVE_RAW / filename
    print(f"[{i:02d}/{len(daily_dates)}] {dt}", end="  ")
    ok = download_file("OA-22299", "1", seq, out_path, REFERER_LIVING, filename)
    if not ok:
        failed_daily.append(filename)
    time.sleep(0.5)

print(f"\n3월 일별 완료. 실패: {len(failed_daily)}개")
if failed_daily:
    print("실패 파일:", failed_daily)

=== 2026년 3월 일별 다운로드 ===
[01/30] 2026-03-01    downloading: seoul_purpose_admdong4_in_20260301.zip ... OK (43.4 MB)
[02/30] 2026-03-02    downloading: seoul_purpose_admdong4_in_20260302.zip ... OK (36.0 MB)
[03/30] 2026-03-03    downloading: seoul_purpose_admdong4_in_20260303.zip ... OK (50.0 MB)
[04/30] 2026-03-04    downloading: seoul_purpose_admdong4_in_20260304.zip ... OK (50.8 MB)
[05/30] 2026-03-05    downloading: seoul_purpose_admdong4_in_20260305.zip ... OK (50.7 MB)
[06/30] 2026-03-06    downloading: seoul_purpose_admdong4_in_20260306.zip ... OK (51.2 MB)
[07/30] 2026-03-07    downloading: seoul_purpose_admdong4_in_20260307.zip ... OK (46.2 MB)
[08/30] 2026-03-08    downloading: seoul_purpose_admdong4_in_20260308.zip ... OK (39.3 MB)
[09/30] 2026-03-09    downloading: seoul_purpose_admdong4_in_20260309.zip ... OK (49.8 MB)
[10/30] 2026-03-10    downloading: seoul_purpose_admdong4_in_20260310.zip ... OK (50.9 MB)
[11/30] 2026-03-11    downloading: seoul_purpose_admdong4_in_2026

### Step 4-4. 성·연령별 도착지 기준 월별 파일 다운로드

10대 미만, 10대, 20대, 30대, 40대, 50대, 60대, 70대 이상을 분리해서 보기 위한 `OA-22298` 월별 파일입니다.


In [ ]:
# ── 4-4. 성·연령별 도착지 기준 월별 파일 (OA-22298) ───────────────
REFERER_AGE_DEST = "https://data.seoul.go.kr/dataList/OA-22298/F/1/datasetView.do"
AGE_DEST_MONTHS = ["202603"]  # 필요하면 ["202501", ..., "202603"] 형태로 추가

print("=== 성·연령별 도착지 기준 월별 파일 다운로드 ===")
failed_age_dest = []
for i, yyyymm in enumerate(AGE_DEST_MONTHS, 1):
    filename = f"seoul_purpose_admdong1_in_{yyyymm}.zip"
    out_path = DRIVE_RAW / filename
    print(f"[{i:02d}/{len(AGE_DEST_MONTHS)}] {yyyymm}", end="  ")
    ok = download_file("OA-22298", "1", yyyymm, out_path, REFERER_AGE_DEST, filename)
    if not ok:
        failed_age_dest.append(filename)
    time.sleep(0.5)

print(f"\n성·연령별 월별 파일 완료. 실패: {len(failed_age_dest)}개")
if failed_age_dest:
    print("실패 파일:", failed_age_dest)


In [9]:
# 다운로드 현황 요약
files = list(DRIVE_RAW.iterdir())
total_mb = sum(f.stat().st_size for f in files) / 1_048_576
print(f"Drive raw/ 파일 수: {len(files)}개")
print(f"총 용량: {total_mb:.0f} MB ({total_mb/1024:.1f} GB)")

Drive raw/ 파일 수: 72개
총 용량: 3122 MB (3.0 GB)


In [ ]:
# ── 4-4. 기존 Drive ZIP 파일 유효성 검사 (corrupt 파일 삭제) ─────────
# Drive에 이미 있던 파일 중 HTML 오류 페이지로 저장된 것을 제거합니다.
print("=== Drive raw/ ZIP 유효성 검사 ===")
corrupt = []
for f in sorted(DRIVE_RAW.glob("*.zip")):
    if not _is_valid_zip(f):
        print(f"  CORRUPT → 삭제: {f.name}")
        f.unlink()
        corrupt.append(f.name)

if corrupt:
    print(f"\n{len(corrupt)}개 corrupt 파일 삭제됨. 위 파일들을 재다운로드하려면")
    print("다운로드 셀들(4-1 ~ 4-3)을 다시 실행하세요.")
else:
    print("모든 ZIP 파일이 유효합니다.")

## Step 4-5. 필수 데이터 다운로드 (매출 · 생활인구)

이 단계는 필수입니다. 파일이 없으면 분석 스크립트가 중단됩니다.

- **매출 데이터** (OA-22175): 행정동별 추정 매출액. 단순 통행지 vs 실소비지 구분에 사용.
- **생활인구** (OA-14991): 행정동별 시간대별 추정 생활인구. 낮/심야 2030 비율로 유동 외부 인구 측정.

> OA-22175의 현재 최신 연도 파일은 2024년 `seq=6`, OA-14991의 현재 최신 월 파일은 2026년 4월 `seq=2604`입니다. 서울 열린데이터광장 파일 목록이 바뀌면 seq를 갱신하세요.


In [ ]:
# ── 4-5a. 상권 추정매출 (OA-22175) ─────────────────────────────────
import zipfile

SALES_OUTPUT = DRIVE_RAW / "seoul_commercial_sales_latest.csv"
SALES_ZIP = DRIVE_RAW / "seoul_commercial_sales_latest.zip"
SALES_REFERER = "https://data.seoul.go.kr/dataList/OA-22175/A/1/datasetView.do"

# OA-22175 파일 다운로드 목록 기준: seq=6은 2024년 파일
SALES_INF_ID  = "OA-22175"
SALES_INF_SEQ = "3"
SALES_SEQ     = "6"

def extract_first_csv_from_zip(zip_path: Path, output_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if not csv_names:
            raise RuntimeError(f"ZIP 내부 CSV가 없습니다: {zip_path}")
        with zf.open(csv_names[0]) as src, open(output_path, "wb") as dst:
            dst.write(src.read())

if SALES_OUTPUT.exists() and SALES_OUTPUT.stat().st_size > 1000:
    print(f"skip existing: {SALES_OUTPUT.name}")
else:
    ok = download_file(SALES_INF_ID, SALES_INF_SEQ, SALES_SEQ,
                       SALES_ZIP, SALES_REFERER, "매출 데이터", is_zip=True)
    if not ok:
        raise RuntimeError("매출 데이터 다운로드 실패")
    extract_first_csv_from_zip(SALES_ZIP, SALES_OUTPUT)
    SALES_ZIP.unlink(missing_ok=True)
    print(f"매출 CSV 추출 완료: {SALES_OUTPUT.name}")

print("매출 데이터 준비 완료")


In [ ]:
# ── 4-5b. 서울 생활인구 행정동 (OA-14991) ──────────────────────────
# 최신 월 파일은 약 40~50 MB 규모입니다.
import zipfile

POP_OUTPUT  = DRIVE_RAW / "seoul_living_population_latest.csv"
POP_ZIP     = DRIVE_RAW / "seoul_living_population_latest.zip"
POP_REFERER = "https://data.seoul.go.kr/dataList/OA-14991/A/1/datasetView.do"

# OA-14991 파일 다운로드 목록 기준: seq=2604는 2026년 4월 파일
POP_INF_ID  = "OA-14991"
POP_INF_SEQ = "3"
POP_SEQ     = "2604"

def extract_first_csv_from_zip(zip_path: Path, output_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        if not csv_names:
            raise RuntimeError(f"ZIP 내부 CSV가 없습니다: {zip_path}")
        with zf.open(csv_names[0]) as src, open(output_path, "wb") as dst:
            dst.write(src.read())

if POP_OUTPUT.exists() and POP_OUTPUT.stat().st_size > 1_000_000:
    print(f"skip existing: {POP_OUTPUT.name}")
else:
    ok = download_file(POP_INF_ID, POP_INF_SEQ, POP_SEQ,
                       POP_ZIP, POP_REFERER, "생활인구 데이터", is_zip=True)
    if not ok:
        raise RuntimeError("생활인구 데이터 다운로드 실패")
    extract_first_csv_from_zip(POP_ZIP, POP_OUTPUT)
    POP_ZIP.unlink(missing_ok=True)
    print(f"생활인구 CSV 추출 완료: {POP_OUTPUT.name}")

print("생활인구 데이터 준비 완료")


## Step 4-6. 필수 GIS · 좌표 · 법정동 파일 준비

아래 파일은 최종 분석 필수 입력입니다. 직접 다운로드 URL을 알고 있으면 `REQUIRED_FILE_URLS`에 넣으면 되고, 비워두면 Colab 업로드 창으로 파일을 올립니다. 파일명은 표에 나온 이름과 정확히 맞추세요.

| 파일명 | 저장 위치 | 용도 |
|---|---|---|
| `seoul_land_use_zone.zip` | Drive raw | 용도지역 비율 |
| `seoul_admin_dong_boundary.zip` | Drive raw | 행정동 경계 공간 조인 |
| `subway_station_coordinates.csv` | Drive raw | 지하철역 좌표 조인 |
| `bus_stop_coordinates.csv` | Drive raw | 버스정류장 좌표 조인 |
| `bjdong_admdong_mapping.csv` | repo metadata | 행정동-법정동 공식 매핑 |


In [ ]:
# ── 4-6. 필수 GIS/좌표/법정동 파일 준비 ─────────────────────────────
from pathlib import Path
import os
import shutil
import requests

REPO_METADATA = Path('/content/seoul_mobility/data_archive/metadata')
REPO_METADATA.mkdir(parents=True, exist_ok=True)

REQUIRED_MANUAL_FILES = {
    'seoul_land_use_zone.zip': DRIVE_RAW / 'seoul_land_use_zone.zip',
    'seoul_admin_dong_boundary.zip': DRIVE_RAW / 'seoul_admin_dong_boundary.zip',
    'subway_station_coordinates.csv': DRIVE_RAW / 'subway_station_coordinates.csv',
    'bus_stop_coordinates.csv': DRIVE_RAW / 'bus_stop_coordinates.csv',
    'bjdong_admdong_mapping.csv': REPO_METADATA / 'bjdong_admdong_mapping.csv',
}

# 직접 다운로드 URL이 있으면 여기에 입력하세요. 비워두면 partial 모드로 계속 실행합니다.
REQUIRED_FILE_URLS = {
    'seoul_land_use_zone.zip': '',
    'seoul_admin_dong_boundary.zip': '',
    'subway_station_coordinates.csv': '',
    'bus_stop_coordinates.csv': '',
    'bjdong_admdong_mapping.csv': '',
}

def _download_direct(url, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    print(f'  download: {target.name}')
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with target.open('wb') as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print('=== 필수 GIS/좌표/법정동 파일 준비 ===')
for filename, target in REQUIRED_MANUAL_FILES.items():
    if target.exists() and target.stat().st_size > 0:
        print(f'  skip existing: {filename}')
        continue
    url = REQUIRED_FILE_URLS.get(filename, '').strip()
    if url:
        _download_direct(url, target)

missing = [name for name, target in REQUIRED_MANUAL_FILES.items() if not target.exists() or target.stat().st_size == 0]
if missing:
    print('\n아래 선택 파일이 없습니다:')
    for name in missing:
        print(' -', name)
    print('\n업로드하지 않아도 파이프라인은 SEOUL_ALLOW_PARTIAL=1로 계속 실행합니다.')
    print('단, GIS 용도지역 비율, 역/정류장 공간결합, 공식 법정동 매핑은 결과에서 제외 또는 근사 처리됩니다.')
    print('최종 제출용 전체 분석이 필요하면 위 파일명을 Drive raw 또는 repo metadata 경로에 배치한 뒤 이 셀을 다시 실행하세요.')
    os.environ['SEOUL_ALLOW_PARTIAL'] = '1'
else:
    os.environ.pop('SEOUL_ALLOW_PARTIAL', None)
    print('필수 GIS/좌표/법정동 파일 준비 완료')

print('SEOUL_ALLOW_PARTIAL =', os.environ.get('SEOUL_ALLOW_PARTIAL', '0'))


## Step 5. 분석 실행

분석 실행 전에 필수 입력 파일을 먼저 검사합니다. 누락 파일이 있으면 대용량 생활이동 ZIP을 읽기 전에 중단됩니다.

In [10]:
import os
from pathlib import Path
os.environ["SEOUL_RAW_DIR"] = str(DRIVE_RAW)
os.environ["SEOUL_OUTPUT_DIR"] = str(DRIVE_OUTPUT)

required_inputs = {
    "매출 데이터": DRIVE_RAW / "seoul_commercial_sales_latest.csv",
    "생활인구 데이터": DRIVE_RAW / "seoul_living_population_latest.csv",
}
optional_full_inputs = {
    "용도지역": DRIVE_RAW / "seoul_land_use_zone.zip",
    "행정동 경계": DRIVE_RAW / "seoul_admin_dong_boundary.zip",
    "지하철역 좌표": DRIVE_RAW / "subway_station_coordinates.csv",
    "버스정류장 좌표": DRIVE_RAW / "bus_stop_coordinates.csv",
    "법정동 매핑": Path("/content/seoul_mobility/data_archive/metadata/bjdong_admdong_mapping.csv"),
}
missing_required = [f"{label}: {path}" for label, path in required_inputs.items() if not path.exists()]
if missing_required:
    raise FileNotFoundError("필수 입력 파일 누락:\n" + "\n".join(missing_required))

missing_optional = [f"{label}: {path}" for label, path in optional_full_inputs.items() if not path.exists()]
if missing_optional:
    os.environ["SEOUL_ALLOW_PARTIAL"] = "1"
    print("부분 실행 모드: 선택 전체분석 파일이 없어 GIS/좌표/공식 법정동 레이어를 건너뜁니다.")
    print("\n".join(missing_optional))
else:
    os.environ.pop("SEOUL_ALLOW_PARTIAL", None)

!python3 /content/seoul_mobility/seoul_mobility_analysis.py


0. Reading administrative-dong mapping...
   saved: /content/seoul_mobility/output/processed/admin_dong_mapping.csv (425 rows)
0-1. Reading 2030 single-household residential data...
   saved: /content/seoul_mobility/output/processed/young_single_household_residential_summary.csv (424 rows)
1. Cleaning subway data...
   saved: /content/seoul_mobility/output/processed/subway_station_daily.csv (18,522 rows)
2. Cleaning bus data...
   saved: /content/seoul_mobility/output/processed/bus_stop_route_summary.csv (42,096 rows)
   saved: /content/seoul_mobility/output/processed/bus_stop_route_hourly.csv (968,208 rows)
3. Analyzing 2030 living-migration OD data...
   reading living migration: seoul_purpose_admdong4_in_20260301.zip
   reading living migration: seoul_purpose_admdong4_in_20260302.zip
   reading living migration: seoul_purpose_admdong4_in_20260303.zip
   reading living migration: seoul_purpose_admdong4_in_20260304.zip
   reading living migration: seoul_purpose_admdong4_in_20260305.zi

## Step 6. 결과를 Drive에 저장

In [ ]:
processed_dir = DRIVE_OUTPUT / "processed"
report_dir = DRIVE_OUTPUT / "reports"

if not processed_dir.exists():
    raise FileNotFoundError(
        f"Drive 결과 폴더가 없습니다: {processed_dir}\n"
        "Step 5 분석 실행 셀이 정상 종료되었는지 확인하세요."
    )

print("분석 결과는 Drive에 바로 저장되어 있습니다:", DRIVE_OUTPUT)
print("processed 파일 수:", len(list(processed_dir.glob("*.csv"))))
print("reports 파일 수:", len(list(report_dir.glob("*"))) if report_dir.exists() else 0)


## Step 7. 시각화

월별 추세, 히트맵, 순위 변화, candidate_type 분포 등을 `seoul_mobility_visualize.py`로 생성합니다. 결과 PNG는 Drive의 `output/reports/viz/run_###/`에 매 실행마다 새로 저장됩니다.


In [ ]:
import os
from pathlib import Path
import pandas as pd

OUTPUT = Path(os.environ.get('SEOUL_OUTPUT_DIR') or globals().get('DRIVE_OUTPUT', Path('/content/drive/MyDrive/seoul_mobility/output')))
if not str(OUTPUT).startswith('/content/drive/'):
    raise RuntimeError(f'Colab Drive output 경로가 아닙니다: {OUTPUT}')

# 한글 폰트가 없으면 차트는 저장되지만 한글 라벨이 깨질 수 있어 Colab에서 먼저 설치합니다.
!apt-get install -y -q fonts-nanum 2>/dev/null
!python3 /content/seoul_mobility/seoul_mobility_visualize.py --output-dir "{OUTPUT}"

VIZ_DIR = OUTPUT / "reports" / "viz"
pngs = sorted(VIZ_DIR.glob("run_*/*.png"))
print(f'차트 {len(pngs)}개가 Drive에 저장되어 있습니다:', VIZ_DIR)
for png in pngs:
    print(' ', png.name)

def _read_output_csv(name, **kwargs):
    path = OUTPUT / "processed" / name
    if not path.exists():
        print(f'[skip] 결과 파일 없음: {path.name}')
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)

monthly = _read_output_csv("monthly_living_migration_2030_summary.csv", dtype={"d_admdong_cd": str})
dest_df = _read_output_csv("living_migration_2030_destination_summary.csv", dtype={"d_admdong_cd": str})
bjdong_df = _read_output_csv("bjdong_candidate_summary.csv", dtype={"bjdong_cd": str})

if monthly.empty or "yyyymm" not in monthly.columns:
    months_sorted = []
    latest_month = None
else:
    monthly["yyyymm"] = monthly["yyyymm"].astype(str)
    months_sorted = sorted(monthly["yyyymm"].dropna().unique())
    latest_month = months_sorted[-1] if months_sorted else None


## 결과 요약

In [ ]:
# ── 결과 요약 ─────────────────────────────────────────────────────
try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

if monthly.empty or latest_month is None:
    print('결과 요약: Step 5(분석 실행)를 먼저 완료하세요.')
else:
    print('===== 분석 결과 요약 =====')
    print(f'분석 기간: {months_sorted[0]} ~ {months_sorted[-1]} ({len(months_sorted)}개월)')
    print(f'분석 행정동 수: {monthly["d_admdong_cd"].nunique()}개')
    latest_m_data = monthly[monthly['yyyymm'] == latest_month]
    for label, filt in [
        ('방문성 검토', '방문성 검토'),
        ('혼재형 (상권+거주)', '혼재형 (상권+거주)'),
        ('2030 거주성 높음', '2030 자취/거주성 높음'),
    ]:
        n = (latest_m_data['residential_filter'] == filt).sum()
        print(f'  {label}: {n}개 행정동')
    if not dest_df.empty and 'visit_pattern_type' in dest_df.columns:
        print('\n방문 패턴 유형 분포:')
        for vpt, cnt in dest_df['visit_pattern_type'].value_counts().items():
            print(f'  {vpt}: {cnt}개 행정동')
    name_col_s = 'd_admdong_name' if 'd_admdong_name' in monthly.columns else 'd_admdong_cd'
    summary_cols = [name_col_s, 'residential_filter', 'adjusted_mobility_score', 'candidate_type']
    candidates = latest_m_data[
        latest_m_data['residential_filter'].isin(['방문성 검토', '혼재형 (상권+거주)'])
    ].sort_values('adjusted_mobility_score', ascending=False)
    print(f'\n방문성 후보 Top 5 ({latest_month}):')
    display(candidates.head(5)[summary_cols].reset_index(drop=True))
    print(f'\n방문성 후보 Bottom 5 ({latest_month}):')
    display(candidates.tail(5).sort_values('adjusted_mobility_score')[summary_cols].reset_index(drop=True))
    if not bjdong_df.empty:
        _sc = 'commercial_potential_score' if 'commercial_potential_score' in bjdong_df.columns else 'adjusted_mobility_score'
        _nm = 'bjdong_nm' if 'bjdong_nm' in bjdong_df.columns else bjdong_df.columns[0]
        print(f'\n법정동 Top 5 ({_sc} 기준):')
        display(bjdong_df.nlargest(5, _sc)[[_nm, _sc]].reset_index(drop=True))
        print(f'\n법정동 Bottom 5 ({_sc} 기준):')
        display(bjdong_df.nsmallest(5, _sc)[[_nm, _sc]].reset_index(drop=True))
